# 🔐 PhishSense AI — Flask API

الباك اند بيبعتلنا URL → بنعمل Feature Extraction → XGBoost بيتنبأ → بنرجع النتيجة

---
### 📋 الـ Cells:
| Cell | الوظيفة |
|------|----------|
| 1 | Install المكتبات |
| 2 | Imports & App Setup |
| 3 | Load الموديل |
| 4 | Feature Extraction Functions |
| 5 | build_features() |
| 6 | API Endpoints |
| 7 | تشغيل السيرفر |

## 📦 CELL 1 — Install المكتبات

In [ ]:
# import sys
# !{sys.executable} -m pip install flask flask-cors python-whois requests numpy xgboost scikit-learn -q
# print('✅ All packages installed.')

## 📥 CELL 2 — Imports & App Setup

In [1]:
import re
import socket
import pickle
import logging
import numpy as np
import ipaddress

from flask        import Flask, request, jsonify
from flask_cors   import CORS
from urllib.parse import urlparse
from datetime     import datetime

# ── Optional imports with graceful fallback ─────────────────
try:
    import requests as http_requests
    HTTP_AVAILABLE = True
    print('✅ requests  — available')
except ImportError:
    HTTP_AVAILABLE = False
    print('⚠️  requests  — NOT available (HTML features will default to 1)')

try:
    import whois
    WHOIS_AVAILABLE = True
    print('✅ whois     — available')
except ImportError:
    WHOIS_AVAILABLE = False
    print('⚠️  whois    — NOT available (domain features will default to 1)')

# ── Flask app ────────────────────────────────────────────────
app = Flask(__name__)
CORS(app)   # بيسمح للباك اند يكلمنا من أي origin

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)s  %(message)s'
)
log = logging.getLogger(__name__)

print('\n✅ Flask app created — CORS enabled')
print(f'   HTTP_AVAILABLE  = {HTTP_AVAILABLE}')
print(f'   WHOIS_AVAILABLE = {WHOIS_AVAILABLE}')

✅ requests  — available
✅ whois     — available

✅ Flask app created — CORS enabled
   HTTP_AVAILABLE  = True
   WHOIS_AVAILABLE = True


## 🤖 CELL 3 — Load XGBoost Model

In [3]:
# ⚠️  ضع ملف الموديل في نفس الـ folder زي main.ipynb
MODEL_PATH = 'D:\Graduation_project\Phish-Sense\Machine-Learning\XGBoostClassifier.pickle.dat'

try:
    with open(MODEL_PATH, 'rb') as f:
        model = pickle.load(f)
    print(f'✅ Model loaded successfully from: {MODEL_PATH}')
    print(f'   Model type: {type(model).__name__}')
except FileNotFoundError:
    model = None
    print(f'❌ Model file not found: {MODEL_PATH}')
    print('   → ضع ملف الموديل جنب main.ipynb وشغّل الـ cell تاني')

✅ Model loaded successfully from: D:\Graduation_project\Phish-Sense\Machine-Learning\XGBoostClassifier.pickle.dat
   Model type: XGBClassifier


<>:2: SyntaxWarning: "\G" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\G"? A raw string is also an option.
<>:2: SyntaxWarning: "\G" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\G"? A raw string is also an option.
C:\Users\SSD\AppData\Local\Temp\ipykernel_10988\2651970156.py:2: SyntaxWarning: "\G" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\G"? A raw string is also an option.
  MODEL_PATH = 'D:\Graduation_project\Phish-Sense\Machine-Learning\XGBoostClassifier.pickle.dat'


## ⚙️ CELL 4 — Feature Extraction Functions

> نفس الكود بالظبط من الـ training notebook

In [4]:
# ── Shortening services regex ────────────────────────────────
SHORTENING_SERVICES = (
    r'bit\.ly|goo\.gl|shorte\.st|go2l\.ink|x\.co|ow\.ly|t\.co|tinyurl|tr\.im|is\.gd|cli\.gs|'
    r'yfrog\.com|migre\.me|ff\.im|tiny\.cc|url4\.eu|twit\.ac|su\.pr|twurl\.nl|snipurl\.com|'
    r'short\.to|BudURL\.com|ping\.fm|post\.ly|Just\.as|bkite\.com|snipr\.com|fic\.kr|loopt\.us|'
    r'doiop\.com|short\.ie|kl\.am|wp\.me|rubyurl\.com|om\.ly|to\.ly|bit\.do|t\.co|lnkd\.in|db\.tt|'
    r'qr\.ae|adf\.ly|goo\.gl|bitly\.com|cur\.lv|tinyurl\.com|ow\.ly|bit\.ly|ity\.im|q\.gs|is\.gd|'
    r'po\.st|bc\.vc|twitthis\.com|u\.to|j\.mp|buzurl\.com|cutt\.us|u\.bb|yourls\.org|x\.co|'
    r'prettylinkpro\.com|scrnch\.me|filoops\.info|vzturl\.com|qr\.net|1url\.com|tweez\.me|v\.gd|'
    r'tr\.im|link\.zip\.net'
)

# ── ADDRESS BAR FEATURES ─────────────────────────────────────

# 1. Domain
def getDomain(url):
    domain = urlparse(url).netloc
    if re.match(r'^www\.', domain):
        domain = re.sub(r'^www\.', '', domain)
    return domain

# 2. Have_IP — IP used instead of domain?
def havingIP(url):
    try:
        host = urlparse(url).netloc
        ipaddress.ip_address(host)
        return 1
    except:
        return 0

# 3. Have_At — '@' symbol in URL?
def haveAtSign(url):
    return 1 if '@' in url else 0

# 4. URL_Length — length >= 54?
def getLength(url):
    return 0 if len(url) < 54 else 1

# 5. URL_Depth — number of sub-pages
def getDepth(url):
    s = urlparse(url).path.split('/')
    return sum(1 for j in s if len(j) != 0)

# 6. Redirection — '//' beyond position 7?
def redirection(url):
    pos = url.rfind('//')
    return 1 if pos > 7 else 0

# 7. https_Domain — 'https' in domain part?
def httpDomain(url):
    return 1 if 'https' in urlparse(url).netloc else 0

# 8. TinyURL — URL shortening service used?
def tinyURL(url):
    return 1 if re.search(SHORTENING_SERVICES, url) else 0

# 9. Prefix/Suffix — '-' in domain?
def prefixSuffix(url):
    return 1 if '-' in urlparse(url).netloc else 0

# ── DOMAIN BASED FEATURES ────────────────────────────────────

# 10. Web_Traffic — Alexa deprecated, default = 1
def web_traffic(url):
    return 1

# 11. Domain_Age — age < 6 months → phishing
def domainAge(domain_info):
    try:
        creation_date   = domain_info.creation_date
        expiration_date = domain_info.expiration_date
        if isinstance(creation_date,   list): creation_date   = creation_date[0]
        if isinstance(expiration_date, list): expiration_date = expiration_date[0]
        ageofdomain = abs((expiration_date - creation_date).days)
        return 1 if (ageofdomain / 30) < 6 else 0
    except:
        return 1

# 12. Domain_End — remaining time < 6 months → phishing
def domainEnd(domain):
    if not WHOIS_AVAILABLE:
        return 1
    try:
        w = whois.whois(domain)
        expiration_date = w.expiration_date
        if isinstance(expiration_date, list):
            expiration_date = expiration_date[0]
        today = datetime.now()
        end = abs((expiration_date.replace(tzinfo=None) - today).days)
        return 1 if (end / 30) < 6 else 0
    except:
        return 1

# ── HTML & JS FEATURES ───────────────────────────────────────

# 13. iFrame — invisible iframe detected?
def iframe(html):
    return 1 if '<iframe' in html.lower() else 0

# 14. Mouse_Over — onmouseover fakes status bar?
def mouseOver(html):
    return 1 if 'onmouseover' in html.lower() else 0

# 15. Right_Click — right-click disabled?
def rightClick(html):
    return 1 if 'event.button==2' in html.lower() else 0

# 16. Web_Forwards — too many redirects?
def forwarding(html):
    return 1 if ('window.location' in html.lower()
                 or 'meta http-equiv' in html.lower()) else 0

print('✅ All 16 feature functions defined.')

✅ All 16 feature functions defined.


## 🔧 CELL 5 — build_features()

In [5]:
FEATURE_NAMES = [
    'Have_IP', 'Have_At', 'URL_Length', 'URL_Depth',
    'Redirection', 'https_Domain', 'TinyURL', 'Prefix/Suffix',
    'DNS_Record', 'Web_Traffic', 'Domain_Age', 'Domain_End',
    'iFrame', 'Mouse_Over', 'Right_Click', 'Web_Forwards',
]

def build_features(url: str):
    """
    Extracts all 16 features from a URL.
    Returns: (np.array shape(1,16), features_dict, warning_str|None)
    """
    domain      = getDomain(url)
    html        = ''
    warning     = None
    domain_info = None

    # ── Fetch HTML ───────────────────────────────────────────
    if HTTP_AVAILABLE:
        try:
            resp = http_requests.get(url, timeout=5)
            html = resp.text
        except Exception as e:
            warning = f'Could not fetch URL: {type(e).__name__}'
            log.warning(warning)

    # ── WHOIS ────────────────────────────────────────────────
    dns = 0
    if WHOIS_AVAILABLE:
        try:
            domain_info = whois.whois(domain)
        except:
            dns = 1
    else:
        dns = 1

    # ── Build vector (must match training column order) ──────
    features = [
        havingIP(url),                              # 1.  Have_IP
        haveAtSign(url),                            # 2.  Have_At
        getLength(url),                             # 3.  URL_Length
        getDepth(url),                              # 4.  URL_Depth
        redirection(url),                           # 5.  Redirection
        httpDomain(url),                            # 6.  https_Domain
        tinyURL(url),                               # 7.  TinyURL
        prefixSuffix(url),                          # 8.  Prefix/Suffix
        dns,                                        # 9.  DNS_Record
        web_traffic(url),                           # 10. Web_Traffic
        1 if dns == 1 else domainAge(domain_info),  # 11. Domain_Age
        domainEnd(domain),                          # 12. Domain_End
        iframe(html),                               # 13. iFrame
        mouseOver(html),                            # 14. Mouse_Over
        rightClick(html),                           # 15. Right_Click
        forwarding(html),                           # 16. Web_Forwards
    ]

    features_dict = dict(zip(FEATURE_NAMES, features))
    arr = np.array(features, dtype=float).reshape(1, -1)
    return arr, features_dict, warning

print('✅ build_features() is ready.')

✅ build_features() is ready.


## 🌐 CELL 6 — API Endpoints

In [6]:
# ════════════════════════════════════════════════════════════
#  GET /api/health  —  Health check
# ════════════════════════════════════════════════════════════
@app.route('/api/health', methods=['GET'])
def health():
    return jsonify({
        'status':       'ok',
        'model_loaded': model is not None,
        'whois':        WHOIS_AVAILABLE,
        'http_client':  HTTP_AVAILABLE,
    })


# ════════════════════════════════════════════════════════════
#  POST /api/analyze/url  —  ده الـ endpoint الأساسي
#  الباك اند بيبعتلنا URL → بنرجعله النتيجة
# ════════════════════════════════════════════════════════════
@app.route('/api/analyze/url', methods=['POST'])
def analyze_url():

    # 1️⃣  Parse request
    data = request.get_json(silent=True) or {}
    url  = data.get('url', '').strip()

    if not url:
        return jsonify({'error': 'url field is required'}), 400

    # 2️⃣  Basic URL validation
    if not url.startswith(('http://', 'https://')):
        url = 'https://' + url

    # 3️⃣  Model check
    if model is None:
        return jsonify({'error': 'Model not loaded. Check MODEL_PATH.'}), 503

    # 4️⃣  Feature extraction
    try:
        log.info(f'Analysing: {url}')
        arr, features_dict, warning = build_features(url)
    except Exception as e:
        log.exception('Feature extraction failed')
        return jsonify({'error': f'Feature extraction error: {str(e)}'}), 500

    # 5️⃣  Prediction
    try:
        prediction = int(model.predict(arr)[0])        # 0 = safe, 1 = phishing
        label      = 'phishing' if prediction == 1 else 'safe'

        confidence = {}
        if hasattr(model, 'predict_proba'):
            proba = model.predict_proba(arr)[0]
            confidence = {
                'safe':     round(float(proba[0]) * 100, 2),
                'phishing': round(float(proba[1]) * 100, 2),
            }
    except Exception as e:
        log.exception('Prediction failed')
        return jsonify({'error': f'Prediction error: {str(e)}'}), 500

    # 6️⃣  Build & return response
    response_data = {
        'url':        url,
        'prediction': label,          # 'safe' | 'phishing'
        'label':      prediction,     # 0 | 1
        'confidence': confidence,     # { safe: %, phishing: % }
        'features':   features_dict,  # all 16 feature values
    }
    if warning:
        response_data['warning'] = warning

    log.info(f'Result: {label} | confidence: {confidence}')
    return jsonify(response_data), 200


# ════════════════════════════════════════════════════════════
#  POST /api/analyze/batch  —  تحليل أكتر من URL دفعة واحدة
# ════════════════════════════════════════════════════════════
@app.route('/api/analyze/batch', methods=['POST'])
def analyze_batch():
    data = request.get_json(silent=True) or {}
    urls = data.get('urls', [])

    if not urls or not isinstance(urls, list):
        return jsonify({'error': 'urls field must be a non-empty list'}), 400

    if model is None:
        return jsonify({'error': 'Model not loaded'}), 503

    results = []
    for url in urls:
        url = url.strip()
        if not url.startswith(('http://', 'https://')):
            url = 'https://' + url
        try:
            arr, features_dict, warning = build_features(url)
            prediction = int(model.predict(arr)[0])
            label      = 'phishing' if prediction == 1 else 'safe'
            confidence = {}
            if hasattr(model, 'predict_proba'):
                proba = model.predict_proba(arr)[0]
                confidence = {
                    'safe':     round(float(proba[0]) * 100, 2),
                    'phishing': round(float(proba[1]) * 100, 2),
                }
            results.append({
                'url':        url,
                'prediction': label,
                'label':      prediction,
                'confidence': confidence,
            })
        except Exception as e:
            results.append({'url': url, 'error': str(e)})

    return jsonify({'results': results}), 200


print('✅ All endpoints registered:')
print('   GET  /api/health')
print('   POST /api/analyze/url    ← الأساسي')
print('   POST /api/analyze/batch')

✅ All endpoints registered:
   GET  /api/health
   POST /api/analyze/url    ← الأساسي
   POST /api/analyze/batch


## 🚀 CELL 7 — Run the Server

> بعد ما تشغّل الـ cell دي، السيرفر هيبدأ على:
> ```
> http://localhost:5000
> ```
> الباك اند يبعت على: `POST http://localhost:5000/api/analyze/url`

In [ ]:
import threading

def run_flask():
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

# تشغيل في thread منفصل علشان الـ notebook ميتجمدش
flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()

print('🚀 Flask server is running!')
print('   URL: http://localhost:5000')
print()
print('   GET  http://localhost:5000/api/health')
print('   POST http://localhost:5000/api/analyze/url')
print('   POST http://localhost:5000/api/analyze/batch')
print()
print('⛔ لإيقاف السيرفر: Kernel → Interrupt')

🚀 Flask server is running!
   URL: http://localhost:5000

   GET  http://localhost:5000/api/health
   POST http://localhost:5000/api/analyze/url
   POST http://localhost:5000/api/analyze/batch

⛔ لإيقاف السيرفر: Kernel → Interrupt
 * Serving Flask app '__main__'


 * Debug mode: off


2026-04-26 23:47:55,984  INFO  WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.1.9:5000
2026-04-26 23:47:55,985  INFO  Press CTRL+C to quit


## 🧪 CELL 8 — اختبار محلي (Optional)

شغّل الـ cell دي للتأكد إن كل حاجة شغالة قبل ما الباك اند يتربط.

In [ ]:
import time
import json
import requests as test_client

time.sleep(1)  # استنى السيرفر يبدأ

BASE_URL  = 'http://localhost:5000'

test_urls = [
    'https://www.google.com',
    'http://y0utube.cam/login/verify?id=123456789',
]

# ── Health Check ─────────────────────────────────────────────
print('=' * 55)
print('🏥 Health Check')
try:
    health_resp = test_client.get(f'{BASE_URL}/api/health', timeout=5)
    print(f'   Status : {health_resp.status_code}')
    print(f'   Body   : {health_resp.json()}')
except Exception as e:
    print(f'   ❌ Health check failed: {e}')

# ── Analyze URLs ─────────────────────────────────────────────
print('=' * 55)
print('🔍 URL Analysis')
print('=' * 55)

for test_url in test_urls:
    try:
        resp   = test_client.post(
            f'{BASE_URL}/api/analyze/url',
            json={'url': test_url},
            timeout=30
        )
        result = resp.json()
        pred   = result.get('prediction', 'error').upper()
        conf   = result.get('confidence', {})
        icon   = '✅' if pred == 'SAFE' else '🚨'

        print(f'{icon}  {pred}')
        print(f'   URL        : {test_url}')
        if conf:
            print(f'   Confidence : Safe {conf.get("safe", "?")}%  |  Phishing {conf.get("phishing", "?")}%')
        if 'warning' in result:
            print(f'   Warning    : {result["warning"]}')
        print('-' * 55)

    except Exception as e:
        print(f'❌ Error testing {test_url}')
        print(f'   {type(e).__name__}: {e}')
        print('-' * 55)

🏥 Health Check
   Status : 200
   Body   : {'service': 'Phish-Sense Backend', 'status': 'healthy', 'timestamp': '2026-04-26T23:51:59.304202', 'version': '1.0.0'}
🔍 URL Analysis
🚨  ERROR
   URL        : https://www.google.com
❌ Error testing https://www.google.com
   AttributeError: 'int' object has no attribute 'get'
-------------------------------------------------------
🚨  ERROR
   URL        : http://y0utube.cam/login/verify?id=123456789
❌ Error testing http://y0utube.cam/login/verify?id=123456789
   AttributeError: 'int' object has no attribute 'get'
-------------------------------------------------------


: 